# AI basics lab 2: tool calling and agents

Run the cells top to bottom with Shift+Enter.

In [1]:
import os

from ai_basics.handmade import HANDMADE_SYSTEM, parse_tool_call
from ai_basics.llm import FakeClient, OllamaClient
from ai_basics.tools import TOOLS

# Which model to talk to. The defaults use your local Ollama.
# Set AI_BASICS_PROVIDER=fake to run everything without a model (this is what CI does).
provider = os.environ.get("AI_BASICS_PROVIDER", "ollama")
model = os.environ.get("AI_BASICS_MODEL", "qwen2.5:3b")

real_client = None if provider == "fake" else OllamaClient(model=model)


def model_or_fake(*script):
    """The real model, or (with the fake provider) a FakeClient that plays back `script`."""
    return FakeClient(list(script)) if provider == "fake" else real_client


QUESTION = "How many students per section are in CS 101?"
print(f"provider: {provider} | model: {model}")

provider: ollama | model: qwen2.5:3b


## Part A. Tool calling by hand

In [2]:
print(HANDMADE_SYSTEM)
print("-" * 60)

# With the fake provider, this is the reply the fake plays back. A real model ignores it.
client = model_or_fake('{"tool": "get_course_info", "arguments": {"course": "CS 101"}}')
MESSAGES = [
    {"role": "system", "content": HANDMADE_SYSTEM},
    {"role": "user", "content": QUESTION},
]

reply = client.chat(MESSAGES).text
print("raw reply:", repr(reply))

call = parse_tool_call(reply)
print("parsed:   ", call)
if call:
    # The model only wrote a string. This line is where YOUR code runs the function.
    print("result:   ", TOOLS[call.name](**call.arguments))

You can use two tools.

Tool: get_course_info
  Argument: course (text, like "CS 101")
  Returns: the course title, number of students and number of sections.

Tool: calculate
  Argument: expression (text, like "84 / 3")
  Returns: the result. Supports + - * / and parentheses. Use it for every calculation.

To use a tool, reply with ONLY this JSON and nothing else:
{"tool": "<tool name>", "arguments": {"<argument name>": "<value>"}}

When you have the final answer, reply with ONLY this JSON and nothing else:
{"answer": "<your answer>"}

Never write any text outside the JSON.
------------------------------------------------------------
raw reply: '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}'
parsed:    ToolCall(name='get_course_info', arguments={'course': 'CS 101'})
result:    {'course': 'CS 101', 'title': 'Intro to Programming', 'students': 84, 'sections': 3}


In [3]:
# Same question 5 times at temperature 1.0. seed=i makes each try repeatable.
# The fake's script: 3 good replies, then a chatty one and a guessed answer.
client = model_or_fake(
    '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}',
    '```json\n{"tool": "get_course_info", "arguments": {"course": "CS 101"}}\n```',
    '{"tool": "get_course_info", "arguments": {"course": "cs101"}}',
    'Sure! {"tool": "get_course_info", "arguments": {"course": "CS 101"}}',
    '{"answer": "28"}',
)

parsed = 0
for i in range(5):
    reply = client.chat(MESSAGES, temperature=1.0, seed=i).text
    ok = parse_tool_call(reply) is not None
    parsed += ok
    print(f"{'OK  ' if ok else 'FAIL'}  {reply[:70]!r}")
print(f"parsed {parsed} of 5")

OK    '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}'
OK    '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}'
OK    '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}'
OK    '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}'
OK    '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}'
parsed 5 of 5


**What I learned (Part A):**

- 